In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel
from imblearn.over_sampling import SMOTE
from sklearn.metrics import classification_report, roc_auc_score, precision_recall_curve, auc, accuracy_score, precision_score, recall_score, f1_score
import numpy as np

# Load the dataset
df = pd.read_csv('../../data/raw/FraudTest.csv')

# Feature Engineering: Extracting features from the transaction date and time
df['trans_date_trans_time'] = pd.to_datetime(df['trans_date_trans_time'])
df['hour'] = df['trans_date_trans_time'].dt.hour
df['day'] = df['trans_date_trans_time'].dt.day
df['day_of_week'] = df['trans_date_trans_time'].dt.dayofweek
df['month'] = df['trans_date_trans_time'].dt.month

# Feature Engineering: Calculating distance between cardholder and merchant locations
def calculate_distance(row):
    lat1, lon1, lat2, lon2 = map(np.radians, [row['lat'], row['long'], row['merch_lat'], row['merch_long']])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    distance = 6371 * c  # Radius of Earth in kilometers
    return distance

df['distance'] = df.apply(calculate_distance, axis=1)

# Identifying and Removing Outliers using IQR
def remove_outliers(df, features):
    for feature in features:
        Q1 = df[feature].quantile(0.25)
        Q3 = df[feature].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        df = df[(df[feature] >= lower_bound) & (df[feature] <= upper_bound)]
    return df

# Removing outliers from numerical features
numerical_features = ['amt', 'city_pop', 'hour', 'day', 'day_of_week', 'month', 'distance']
df = remove_outliers(df, numerical_features)

# Selecting relevant features for the model
features = ['amt', 'city_pop', 'hour', 'day', 'day_of_week', 'month', 'distance',
            'merchant', 'category', 'state', 'job', 'gender', 'city']
X = df[features]
y = df['is_fraud']

# Splitting the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

categorical_features = ['merchant', 'category', 'state', 'job', 'gender', 'city']

numerical_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Preprocessing the training data
X_train_preprocessed = preprocessor.fit_transform(X_train)
X_test_preprocessed = preprocessor.transform(X_test)

# Function to evaluate model performance with different numbers of features
def evaluate_features(X_train, y_train, X_test, y_test, num_features_list):
    results = {}
    for num_features in num_features_list:
        selector = SelectFromModel(RandomForestClassifier(n_estimators=20, random_state=42), max_features=num_features)
        X_train_selected = selector.fit_transform(X_train, y_train)
        X_test_selected = selector.transform(X_test)
        
        # Resampling the training data using SMOTE
        smote = SMOTE(random_state=42)
        X_resampled, y_resampled = smote.fit_resample(X_train_selected, y_train)

        # Training a RandomForestClassifier with 20 estimators
        rf = RandomForestClassifier(n_estimators=20, random_state=42)
        rf.fit(X_resampled, y_resampled)

        # Making predictions on the test set
        y_pred = rf.predict(X_test_selected)
        y_pred_prob = rf.predict_proba(X_test_selected)[:, 1]

        # Evaluating the model
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        roc_auc = roc_auc_score(y_test, y_pred_prob)
        
        precision_vals, recall_vals, _ = precision_recall_curve(y_test, y_pred_prob)
        pr_auc = auc(recall_vals, precision_vals)
        
        results[num_features] = {
            'Accuracy': accuracy,
            'Precision': precision,
            'Recall': recall,
            'F1 Score': f1,
            'ROC-AUC': roc_auc,
            'PR AUC': pr_auc,
            'Classification Report': classification_report(y_test, y_pred)
        }
        
        print(f'Num features: {num_features} | Accuracy: {accuracy:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f} | F1 Score: {f1:.4f} | ROC-AUC: {roc_auc:.4f} | PR AUC: {pr_auc:.4f}')
        print(results[num_features]['Classification Report'])
    
    return results

# Example usage
num_features_list = [30, 50, 100, 150, 200]
results = evaluate_features(X_train_preprocessed, y_train, X_test_preprocessed, y_test, num_features_list)


Num features: 30 | Accuracy: 0.9991 | Precision: 0.6304 | Recall: 0.3118 | F1 Score: 0.4173 | ROC-AUC: 0.8769 | PR AUC: 0.3964
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     85566
           1       0.63      0.31      0.42        93

    accuracy                           1.00     85659
   macro avg       0.81      0.66      0.71     85659
weighted avg       1.00      1.00      1.00     85659

Num features: 50 | Accuracy: 0.9990 | Precision: 0.6087 | Recall: 0.3011 | F1 Score: 0.4029 | ROC-AUC: 0.8714 | PR AUC: 0.3881
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     85566
           1       0.61      0.30      0.40        93

    accuracy                           1.00     85659
   macro avg       0.80      0.65      0.70     85659
weighted avg       1.00      1.00      1.00     85659

Num features: 100 | Accuracy: 0.9991 | Precision: 0.7143 | Recall: 0.2688 | F1 Score: 0.3906